In [3]:
# =============================================================================
# GSE24080 DATASET DOWNLOAD
# =============================================================================
# GSE24080 contains gene expression profiles of 559 Multiple Myeloma patients
# from the APEX clinical trial, with overall survival and event-free survival
# data. Data was generated using Affymetrix Human Genome U133Plus2 arrays.
#
# Reference: Mulligan et al., Blood 2007




# =============================================================================

import GEOparse
import pandas as pd
import numpy as np

# Download GSE24080 from GEO
# This will download the dataset to the current directory
print("Downloading GSE24080 from GEO... (this may take a few minutes)")

# THIS DOES NOT WORK FROM MY PC, so I manually downladed the SOFT data file
# from: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE24080
#gse = GEOparse.get_GEO(geo="GSE24080", destdir="../data/")

gse = GEOparse.get_GEO(filepath="../data/GSE24080_family.soft.gz")

print(f"\nDataset loaded successfully")
print(f"GSE name: {gse.name}")
print(f"Number of samples: {len(gse.gsms)}")
print(f"Number of platforms: {len(gse.gpls)}")

04-Mar-2026 09:52:01 INFO GEOparse - Parsing ../data/GSE24080_family.soft.gz: 
04-Mar-2026 09:52:01 DEBUG GEOparse - DATABASE: GeoMiame
04-Mar-2026 09:52:01 DEBUG GEOparse - SERIES: GSE24080
04-Mar-2026 09:52:01 DEBUG GEOparse - PLATFORM: GPL570


/home/miguel/myeloma-survival-genomics/venv/lib/python3.13/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
04-Mar-2026 09:52:02 DEBUG GEOparse - SAMPLE: GSM592391
04-Mar-2026 09:52:02 DEBUG GEOparse - SAMPLE: GSM592392
04-Mar-2026 09:52:02 DEBUG GEOparse - SAMPLE: GSM592393
04-Mar-2026 09:52:02 DEBUG GEOparse - SAMPLE: GSM592394
04-Mar-2026 09:52:02 DEBUG GEOparse - SAMPLE: GSM592395
04-Mar-2026 09:52:02 DEBUG GEOparse - SAMPLE: GSM592396
04-Mar-2026 09:52:02 DEBUG GEOparse - SAMPLE: GSM592397
04-Mar-2026 09:52:03 DEBUG GEOparse - SAMPLE: GSM592398
04-Mar-2026 09:52:03 DEBUG GEOparse - SAMPLE: GSM592399
04-Mar-2026 09:52:03 DEBUG GEOparse - SAMPLE: GSM592400
04-Mar-2026 09:52:03 DEBUG GEOparse - SAMPLE: GSM592401
04-Mar-2026 09:52:03 DEBUG GEOparse - SAMPLE: GSM592402
04-Mar-2026 09:52:03 DEBUG GEOparse - SAMPLE: GSM592403
04-Mar-2026 0


Dataset loaded successfully
GSE name: GSE24080
Number of samples: 559
Number of platforms: 1


In [4]:
# =============================================================================
# DATASET STRUCTURE EXPLORATION
# =============================================================================

# Explore metadata of first sample
first_sample_id = list(gse.gsms.keys())[0]
first_sample = gse.gsms[first_sample_id]

print(f"First sample ID: {first_sample_id}")
print(f"\nSample metadata keys:")
for key, value in first_sample.metadata.items():
    print(f"  {key}: {value[:1] if isinstance(value, list) else value}")

print(f"\nExpression data shape: {first_sample.table.shape}")
print(f"\nExpression data preview:")
print(first_sample.table.head())

First sample ID: GSM592391

Sample metadata keys:
  title: ['P0002-01-FAKE03-U133Plus-2']
  geo_accession: ['GSM592391']
  status: ['Public on Sep 11 2010']
  submission_date: ['Sep 10 2010']
  last_update_date: ['Sep 10 2010']
  type: ['RNA']
  channel_count: ['1']
  source_name_ch1: ['pre-treatment bone marrow']
  organism_ch1: ['Homo sapiens']
  taxid_ch1: ['9606']
  characteristics_ch1: ['maqc_distribution_status: Training']
  molecule_ch1: ['total RNA']
  extract_protocol_ch1: ['Total RNA was isolated with RNeasy Mini Kit (Qiagen, Valencia, CA).']
  label_ch1: ['biotin']
  label_protocol_ch1: ['As recommended by manufacturer']
  hyb_protocol: ['As recommended by manufacturer']
  scan_protocol: ['As recommended by manufacturer']
  description: ['MAQC-II multiple myeloma (MM) data set (endpoints F, G, H, and I)']
  data_processing: ['All data used in these analyses were derived with the Affymetrix Microarray Suite GCOS1.1 software.']
  platform_id: ['GPL570']
  contact_name: ['Lemin

In [5]:
# =============================================================================
# EXTRACT CLINICAL/SURVIVAL DATA
# =============================================================================

# Extract metadata from all samples
metadata_list = []
for sample_id, sample in gse.gsms.items():
    meta = {'sample_id': sample_id}
    for key, value in sample.metadata.items():
        meta[key] = value[0] if isinstance(value, list) and len(value) == 1 else value
    metadata_list.append(meta)

df_meta = pd.DataFrame(metadata_list)

print(f"Metadata shape: {df_meta.shape}")
print(f"\nColumns: {list(df_meta.columns)}")
print(f"\ncharacteristics_ch1 unique values (first 20):")
print(df_meta['characteristics_ch1'].value_counts().head(20))

Metadata shape: (559, 34)

Columns: ['sample_id', 'title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_phone', 'contact_laboratory', 'contact_department', 'contact_institute', 'contact_address', 'contact_city', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'series_id', 'data_row_count']

characteristics_ch1 unique values (first 20):
characteristics_ch1
[maqc_distribution_status: Validation, age: 61.0, Sex: male, randomly assigned class label: 1, efs milestone outcome (24 months): 1=event occurred < 24 months, 0=no event in first 24 months: 0, os milestone outcome (24 months): 1= deceased by 24 months, 0= alive at 24 months: 0]      7
[maqc_distribution_

In [7]:
# =============================================================================
# DATASET OVERVIEW
# =============================================================================

print("\n=== COLUMN DESCRIPTIONS ===")
col_descriptions = {
    'sample_id': ('GEO sample accession ID', 'Unique identifier for each patient sample in GEO database'),
    'title': ('Sample title/identifier', 'Internal lab identifier for the sample'),
    'geo_accession': ('GEO accession number', 'Same as sample_id, redundant field'),
    'status': ('Public availability status', 'Whether the data is publicly accessible'),
    'submission_date': ('Date submitted to GEO', 'When the sample was uploaded to GEO'),
    'last_update_date': ('Date of last update', 'Last modification date in GEO'),
    'type': ('Data type', 'Always RNA for gene expression experiments'),
    'channel_count': ('Number of channels', 'Single channel = 1 color array (Affymetrix standard)'),
    'source_name_ch1': ('Tissue source', 'Bone marrow — where myeloma plasma cells reside'),
    'organism_ch1': ('Organism', 'Homo sapiens — human samples'),
    'taxid_ch1': ('NCBI taxonomy ID', '9606 is the taxonomy ID for Homo sapiens'),
    'characteristics_ch1': ('Clinical characteristics', 'Contains age, sex, training/validation split and survival outcomes (EFS, OS at 24 months)'),
    'molecule_ch1': ('Molecule type', 'Total RNA — measures all gene expression in the sample'),
    'extract_protocol_ch1': ('RNA extraction protocol', 'RNeasy Mini Kit (Qiagen) — standard protocol for RNA isolation'),
    'label_ch1': ('Array label type', 'Biotin — used to label RNA for hybridization to the array'),
    'label_protocol_ch1': ('Labeling protocol', 'Manufacturer standard protocol for biotin labeling'),
    'hyb_protocol': ('Hybridization protocol', 'Process of binding labeled RNA to array probes'),
    'scan_protocol': ('Scanning protocol', 'How the array fluorescence signal was measured'),
    'description': ('Sample description', 'MAQC-II MM dataset — part of FDA MicroArray Quality Control project'),
    'data_processing': ('Data processing method', 'Affymetrix GCOS 1.1 — normalizes raw intensity values across arrays'),
    'platform_id': ('Affymetrix platform ID', 'GPL570 = Affymetrix Human Genome U133 Plus 2.0 Array — 54,675 probes covering ~20,000 genes'),
    'contact_name': ('Contact researcher name', 'Principal investigator responsible for the dataset'),
    'contact_email': ('Contact email', 'Email of the principal investigator'),
    'contact_phone': ('Contact phone', 'Phone of the principal investigator'),
    'contact_laboratory': ('Laboratory name', 'Lab where the experiment was conducted'),
    'contact_department': ('Department', 'Academic department of the principal investigator'),
    'contact_institute': ('Institution', 'University or research center'),
    'contact_address': ('Address', 'Physical address of the institution'),
    'contact_city': ('City', 'City of the institution'),
    'contact_zip/postal_code': ('Postal code', 'Postal code of the institution'),
    'contact_country': ('Country', 'Country of the institution'),
    'supplementary_file': ('Raw CEL file', 'FTP path to the raw Affymetrix CEL file for each sample'),
    'series_id': ('GEO series ID', 'Parent study identifier — links this sample to the full GSE24080 study'),
    'data_row_count': ('Number of probes', '54,675 probes measured per sample on the U133Plus2 array')
}

df_desc = pd.DataFrame({
    'Column': col_descriptions.keys(),
    'Short description': [v[0] for v in col_descriptions.values()],
    'Meaning': [v[1] for v in col_descriptions.values()]
})
print(df_desc.to_string(index=False))


=== COLUMN DESCRIPTIONS ===
                 Column          Short description                                                                                     Meaning
              sample_id    GEO sample accession ID                                   Unique identifier for each patient sample in GEO database
                  title    Sample title/identifier                                                      Internal lab identifier for the sample
          geo_accession       GEO accession number                                                          Same as sample_id, redundant field
                 status Public availability status                                                     Whether the data is publicly accessible
        submission_date      Date submitted to GEO                                                         When the sample was uploaded to GEO
       last_update_date        Date of last update                                                               

In [ ]:
# =============================================================================
# PARSE CLINICAL FEATURES FROM characteristics_ch1
# =============================================================================

# =============================================================================
# PARSE CLINICAL FEATURES FROM characteristics_ch1
# =============================================================================
# The SOFT file stores clinical data for each patient as a list of strings
# in the format "key: value" inside the characteristics_ch1 field.
#
# Example for one patient:
#   ['maqc_distribution_status: Training',
#    'age: 53.4',
#    'Sex: female',
#    'randomly assigned class label: 1',
#    'efs milestone outcome (24 months): ... : 0',
#    'os milestone outcome (24 months): ... : 0']
#
# This cell parses those strings into a structured DataFrame where each
# clinical attribute becomes a separate column, one row per patient.
#
# Extracted clinical features:
#   - maqc_distribution_status: Training/Validation split predefined by study
#   - age:                       Patient age at diagnosis
#   - Sex:                       Patient sex (male/female)
#   - randomly assigned class label: Internal study label
#   - efs_24m:                   Event-Free Survival at 24 months (binary: 0/1)
#                                 1 = relapse or progression occurred before 24 months
#                                 0 = no event in first 24 months
#   - os_24m:                    Overall Survival at 24 months (binary: 0/1)
#                                 1 = deceased before 24 months
#                                 0 = alive at 24 months
# =============================================================================

def parse_characteristics(char_list):
    """Parse list of key:value strings into a dict."""
    result = {}
    if not isinstance(char_list, list):
        char_list = [char_list]
    for item in char_list:
        if ':' in item:
            key, _, value = item.partition(':')
            result[key.strip()] = value.strip()
    return result

# Parse all samples
parsed = df_meta['characteristics_ch1'].apply(parse_characteristics)
df_clinical = pd.DataFrame(list(parsed))
df_clinical['sample_id'] = df_meta['sample_id'].values

print(f"Clinical data shape: {df_clinical.shape}")
print(f"\nColumns: {list(df_clinical.columns)}")
print(f"\nPreview:")
print(df_clinical.head())

Clinical data shape: (559, 7)

Columns: ['maqc_distribution_status', 'age', 'Sex', 'randomly assigned class label', 'efs milestone outcome (24 months)', 'os milestone outcome (24 months)', 'sample_id']

Preview:
  maqc_distribution_status   age     Sex randomly assigned class label  \
0                 Training  53.4  female                             1   
1                 Training  52.7  female                             1   
2                 Training  62.0    male                             1   
3                 Training  60.3    male                             1   
4                 Training  66.9    male                             1   

                   efs milestone outcome (24 months)  \
0  1=event occurred < 24 months, 0=no event in fi...   
1  1=event occurred < 24 months, 0=no event in fi...   
2  1=event occurred < 24 months, 0=no event in fi...   
3  1=event occurred < 24 months, 0=no event in fi...   
4  1=event occurred < 24 months, 0=no event in fi...   

      

In [12]:
# =============================================================================
# CLEAN CLINICAL DATA
# =============================================================================
# Rename columns, extract numeric values from OS and EFS,
# and convert types for analysis.
# =============================================================================

df_clean = df_clinical.copy()

# Rename columns
df_clean.columns = [
    'split', 'age', 'sex', 'class_label', 'efs_24m', 'os_24m', 'sample_id'
]

# Extract numeric value (last character after final ':')
df_clean['efs_24m'] = df_clean['efs_24m'].str.split(':').str[-1].str.strip().astype(int)
df_clean['os_24m'] = df_clean['os_24m'].str.split(':').str[-1].str.strip().astype(int)

# Convert types
df_clean['age'] = df_clean['age'].astype(float)
df_clean['class_label'] = df_clean['class_label'].astype(int)
df_clean['sex'] = df_clean['sex'].str.lower()

print(f"Clean clinical data shape: {df_clean.shape}")
print(f"\nData types:\n{df_clean.dtypes}")
print(f"\nPreview:")
print(df_clean.head())
print(f"\nOS 24m distribution:\n{df_clean['os_24m'].value_counts()}")
print(f"\nEFS 24m distribution:\n{df_clean['efs_24m'].value_counts()}")
print(f"\nSplit distribution:\n{df_clean['split'].value_counts()}")

Clean clinical data shape: (559, 7)

Data types:
split           object
age            float64
sex             object
class_label      int64
efs_24m          int64
os_24m           int64
sample_id       object
dtype: object

Preview:
      split   age     sex  class_label  efs_24m  os_24m  sample_id
0  Training  53.4  female            1        0       0  GSM592391
1  Training  52.7  female            1        0       0  GSM592392
2  Training  62.0    male            1        0       0  GSM592393
3  Training  60.3    male            1        0       0  GSM592394
4  Training  66.9    male            1        0       0  GSM592395

OS 24m distribution:
os_24m
0    481
1     78
Name: count, dtype: int64

EFS 24m distribution:
efs_24m
0    441
1    118
Name: count, dtype: int64

Split distribution:
split
Training       340
Validation     214
MAQC_Remove      5
Name: count, dtype: int64


In [14]:
# =============================================================================
# EXTRACT GENE EXPRESSION MATRIX
# =============================================================================
# Build a samples x probes matrix from all 559 samples.
# This will be a large matrix: 559 rows x 54,675 columns.
# This step may take several minutes.
# =============================================================================

# =============================================================================
# EXTRACT GENE EXPRESSION MATRIX
# =============================================================================
# Each patient sample in the SOFT file contains a table with 54,675 rows,
# one per probe, with the corresponding expression intensity value.
#
# This cell iterates over all 559 samples and builds a single matrix where:
#   - Each ROW is a patient (identified by GEO sample ID)
#   - Each COLUMN is a probe (identified by Affymetrix probe ID e.g. '229819_at')
#   - Each VALUE is the log2-normalized expression intensity for that probe
#     in that patient, as processed by Affymetrix GCOS 1.1 software
#
# Result: a 559 x 54,675 matrix — 559 patients x 54,675 probes
# Memory footprint: ~244 MB
#
# Note: probe IDs (e.g. '229819_at') are not yet mapped to gene symbols.
#       This annotation step will be performed in the next cell using the
#       GPL570 platform table (probe ID -> gene symbol mapping).
# =============================================================================


expression_data = {}
for i, (sample_id, sample) in enumerate(gse.gsms.items()):
    expression_data[sample_id] = sample.table.set_index('ID_REF')['VALUE']
    if i % 50 == 0:
        print(f"  Processed {i+1}/559 samples...")

df_expr = pd.DataFrame(expression_data).T
df_expr.index.name = 'sample_id'

print(f"\nExpression matrix shape: {df_expr.shape}")
print(f"Memory usage: {df_expr.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nPreview (5x5):")
print(df_expr.iloc[:5, :5])

  Processed 1/559 samples...
  Processed 51/559 samples...
  Processed 101/559 samples...
  Processed 151/559 samples...
  Processed 201/559 samples...
  Processed 251/559 samples...
  Processed 301/559 samples...
  Processed 351/559 samples...
  Processed 401/559 samples...
  Processed 451/559 samples...
  Processed 501/559 samples...
  Processed 551/559 samples...

Expression matrix shape: (559, 54675)
Memory usage: 244.5 MB

Preview (5x5):
ID_REF     229819_at  217757_at  1558450_at  214440_at  206797_at
sample_id                                                        
GSM592391     4.6004     6.8917      8.2270     9.1160     4.6754
GSM592392     9.1276     5.0407      7.7843     9.4530     5.5673
GSM592393     8.7930     5.8932      8.2629     9.4249     5.4063
GSM592394     9.0998     6.8837      8.5397     8.7931     5.2767
GSM592395     9.2174     5.3384      7.9051    10.3616     4.8507


In [15]:
# =============================================================================
# PROBE TO GENE SYMBOL ANNOTATION
# =============================================================================
# The expression matrix uses Affymetrix probe IDs (e.g. '229819_at') as
# column names. This cell maps those IDs to human gene symbols (e.g. 'TP53')
# using the GPL570 platform annotation table included in the SOFT file.
#
# Key considerations:
#   - Multiple probes can map to the same gene — we keep the probe with the
#     highest mean expression (most informative)
#   - Some probes have no gene annotation — these are discarded
#   - After mapping, columns become gene symbols instead of probe IDs
# =============================================================================

# Get platform annotation table
gpl = gse.gpls['GPL570']
df_annot = gpl.table

print(f"Annotation table shape: {df_annot.shape}")
print(f"Columns: {list(df_annot.columns[:10])}")
print(f"\nPreview:")
print(df_annot[['ID', 'Gene Symbol', 'Gene Title']].head(10))

Annotation table shape: (54675, 16)
Columns: ['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title']

Preview:
          ID       Gene Symbol  \
0  1007_s_at  DDR1 /// MIR4640   
1    1053_at              RFC2   
2     117_at             HSPA6   
3     121_at              PAX8   
4  1255_g_at            GUCA1A   
5    1294_at  MIR5193 /// UBA7   
6    1316_at              THRA   
7    1320_at            PTPN21   
8  1405_i_at              CCL5   
9    1431_at            CYP2E1   

                                          Gene Title  
0  discoidin domain receptor tyrosine kinase 1 //...  
1        replication factor C (activator 1) 2, 40kDa  
2               heat shock 70kDa protein 6 (HSP70B')  
3                                       paired box 8  
4            guanylate cyclase activator 1A (retina)  
5  microRNA 5193 /// ubiquitin-like modifier acti...  
6          

In [18]:
# =============================================================================
# MAP PROBES TO GENE SYMBOLS
# =============================================================================

# Keep only probes with a single unambiguous gene symbol
# If multiple probes map to same gene, keep the one with highest mean expression
# Memory-efficient approach: process gene by gene
print("Deduplicating genes...")

# Find duplicate genes
gene_counts = df_expr_annot.columns.value_counts()
duplicate_genes = gene_counts[gene_counts > 1].index

# For duplicates, keep probe with highest mean expression
cols_to_keep = []
for gene in df_expr_annot.columns.unique():
    gene_cols = df_expr_annot.loc[:, df_expr_annot.columns == gene]
    if gene_cols.shape[1] > 1:
        best_probe = gene_cols.mean(axis=0).idxmax()
        cols_to_keep.append(best_probe)
    else:
        cols_to_keep.append(gene)

# Rebuild matrix with deduplicated columns
df_expr_genes = df_expr_annot.loc[:, ~df_expr_annot.columns.duplicated(keep=False)]
df_expr_dedup = pd.concat([
    df_expr_genes,
    df_expr_annot[duplicate_genes].loc[:, ~df_expr_annot[duplicate_genes].columns.duplicated()]
], axis=1)

print(f"Expression matrix after deduplication: {df_expr_dedup.shape}")
print(f"\nPreview (5x5):")
print(df_expr_dedup.iloc[:5, :5])

Deduplicating genes...
Expression matrix after deduplication: (559, 21655)

Preview (5x5):
ID_REF       A1BG     NAT1    NAT2  SERPINA3   AADAC
sample_id                                           
GSM592391  4.6004   9.1160  4.6754    6.4355  4.8679
GSM592392  9.1276   9.4530  5.5673    5.7468  5.1463
GSM592393  8.7930   9.4249  5.4063    5.1678  4.3810
GSM592394  9.0998   8.7931  5.2767    5.3387  4.0859
GSM592395  9.2174  10.3616  4.8507    5.1855  4.7965


In [19]:
# =============================================================================
# SAVE PROCESSED DATA TO DISK
# =============================================================================

import os
os.makedirs('../data', exist_ok=True)

# Save clinical data
df_clean.to_csv('../data/clinical_data.csv', index=False)
print(f"Clinical data saved: {df_clean.shape}")

# Save expression matrix
df_expr_dedup.to_csv('../data/expression_matrix.csv')
print(f"Expression matrix saved: {df_expr_dedup.shape}")

# Save annotation table
df_annot_clean.to_csv('../data/probe_annotations.csv', index=False)
print(f"Probe annotations saved: {df_annot_clean.shape}")

print("\nAll data saved to data/ ✅")

Clinical data saved: (559, 7)
Expression matrix saved: (559, 21655)
Probe annotations saved: (42986, 2)

All data saved to data/ ✅
